### **Lending Club Loan Default Prediction**

#### 📌 Notebook Scope

This notebook focuses only on:
- Understanding the dataset
- Interpreting features from a business perspective
- Identifying data quality issues

❌ No preprocessing or modeling is done here.
Those steps will be implemented in the pipeline modules.


#### ⚠️ Note:
This notebook analyzes **accepted loans only**.
Rejected loans will be analyzed separately using Power BI
to understand rejection patterns and bias.

#### **Project Overview**

This project focuses on building an end-to-end, industry-style Machine Learning pipeline to analyze and predict loan default risk using the Lending Club dataset. The primary focus is on :

Correct problem framing

Preventing data leakage

Writing clean, modular, production-style code

Making business-driven decisions

#### **Dataset Description**

This project uses the Lending Club Accepted Loans dataset, which includes:

Borrower demographics (income, employment length, home ownership)

Loan details (loan amount, term, interest rate)

Credit history indicators (DTI, credit score proxies)

Loan outcome (loan_status)

**Important Note on Data Usage**
Only features available **at the time of loan application** are used for modeling. Columns related to **hardship, settlements, or post-default** events are excluded to ensure realistic predictions.

#### **Business Goal**

The primary objective of this project is to: Predict whether a loan applicant is likely to default, before the loan is approved.

From a business perspective, this helps financial institutions to:

Reduce credit risk

Improve loan approval decisions

Optimize interest rates

Minimize financial losses

#### **Machine Learning Goal**

From a data science perspective, the goal is to:

Build a binary classification model:

**0 → Loan will not default**

**1 → Loan will default**

Use only pre-loan information (no data leakage)

Evaluate models using:

Precision , Recall , F1-score , ROC-AUC 

In [1]:
import pandas as pd
import numpy as np

In [2]:
import sys
sys.path.append(r'C:\Users\bhagyashree.s\Desktop\Machine_Learning_Projects\LendingClub_End_to_End_ML\src')

In [3]:
from data.load_data import load_dataset

In [4]:
# Loading a manageable sample of data for exploration

df = load_dataset(
    path = r'C:\Users\bhagyashree.s\Desktop\Machine_Learning_Projects\LendingClub_End_to_End_ML\src\data\raw\accepted_2007_to_2018Q4.csv',
    nrows = 100_000
)

In [5]:
# check shape

df.shape

(100000, 151)

In [6]:
pd.set_option('display.max_columns', None)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Columns: 151 entries, id to settlement_term
dtypes: float64(114), int64(1), object(36)
memory usage: 115.2+ MB


In [11]:
 for col in df.columns:
     print(col)

id
member_id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
url
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
fico_range_low
fico_range_high
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
last_fico_range_high
last_fico_range_low
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
to

A bank’s core questions are :

**1. Should we give the loan or not?**

**2. If yes, how risky is this customer?**

**3. What interest rate compensates that risk?**

**4. How much money could we lose if they default?**

#### **Feature Understanding & Business Reasoning (Lending Club Dataset)**


| Feature Name | Feature Meaning | Business Reasoning | Action |
|-------------|----------------|-------------------|--------|
| id | Unique loan identifier | Used only for record tracking, no relation to default risk | Drop |
| member_id | Unique borrower ID | Mostly null and has no predictive value | Drop |
| loan_amnt | Loan amount requested | Higher loan amount increases exposure and default risk | Keep |
| funded_amnt | Amount funded by LendingClub | Known after loan approval → data leakage | Drop |
| funded_amnt_inv | Amount funded by investors | Post-approval information → leakage | Drop |
| term | Loan duration (36/60 months) | Longer tenure increases uncertainty and risk | Keep |
| int_rate | Interest rate | Higher interest reflects higher borrower risk | Keep |
| installment | Monthly EMI | High EMI relative to income increases default risk | Keep |
| grade | LendingClub loan grade | Internal risk classification | Keep |
| sub_grade | Fine-grained loan grade | More precise borrower risk segmentation | Keep |
| emp_title | Borrower job title | Very high cardinality, weak signal | Drop |
| emp_length | Employment length | Longer employment implies income stability | Keep |
| home_ownership | Rent / Own / Mortgage | Asset ownership lowers default probability | Keep |
| annual_inc | Annual income | Primary indicator of repayment capacity | Keep |
| verification_status | Income verification status | Verified income reduces risk | Keep |
| issue_d | Loan issue date | Useful for time-based and trend analysis | Keep |
| loan_status | Loan outcome | Defines default vs non-default | Target |
| pymnt_plan | Payment plan indicator | Rare and low predictive value | Drop |
| purpose | Loan purpose | Certain purposes carry higher risk | Keep |
| title | Loan title | Redundant with purpose | Drop |
| zip_code | Borrower ZIP code | High cardinality, low predictive power | Drop |
| addr_state | Borrower state | Regional economic risk indicator | Keep |
| dti | Debt-to-income ratio | High DTI signals financial stress | Keep |
| delinq_2yrs | Past delinquencies | Strong indicator of risky behavior | Keep |
| earliest_cr_line | Credit history start date | Longer credit history lowers risk | Keep |
| fico_range_low | Lower FICO score | Strongest predictor of default | Keep |
| fico_range_high | Upper FICO score | Complements credit risk assessment | Keep |
| inq_last_6mths | Credit inquiries | Frequent inquiries indicate risk | Keep |
| open_acc | Open credit accounts | Measures current credit exposure | Keep |
| pub_rec | Public derogatory records | Severe negative credit signal | Keep |
| revol_bal | Revolving credit balance | High balance increases repayment pressure | Keep |
| revol_util | Credit utilization ratio | High utilization indicates risk | Keep |
| total_acc | Total credit accounts | Reflects credit experience | Keep |
| out_prncp | Outstanding principal | Known only after loan issuance | Drop |
| total_pymnt | Total payments received | Future information → leakage | Drop |
| recoveries | Amount recovered post-default | Occurs after default | Drop |
| last_pymnt_d | Last payment date | Post-event data | Drop |
| last_pymnt_amnt | Last payment amount | Leakage | Drop |
| hardship_* | Hardship related features | Known only after borrower distress | Drop |
| settlement_* | Settlement related features | Post-default information | Drop |
| debt_settlement_flag | Debt settlement indicator | Happens after default | Drop |





### Business Perspective

The goal of this project is to **predict loan default risk before loan approval**.

Therefore:
- Any feature **created after loan approval or default** is removed to avoid data leakage.
- Features related to **borrower capacity** (income, DTI, installment) are prioritized.
- Features reflecting **lender internal decisions** (grade, sub_grade) are treated carefully.
- Temporal features help identify **economic cycles and seasonal risk patterns**.

This ensures the model mirrors how a **real credit risk system** would operate in production

In [12]:
# Identify numerical columns

numerical_cols = df.select_dtypes(include = ['int64', 'float64']).columns.tolist()
numerical_cols

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'int_rate',
 'installment',
 'annual_inc',
 'dti',
 'delinq_2yrs',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_amnt',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'annual_inc_joint',
 'dti_joint',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m',
 'mths_since_rcnt_il',
 'total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m',
 'max_bal_bc',
 'all_util',
 'total_rev_hi_lim',
 'inq_fi',
 'total_cu_tl',
 'inq_last_12m',
 'acc_open_past_24mths',
 'avg_cu

In [13]:
df.describe()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_amnt,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,annual_inc_joint,dti_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,deferral_term,hardship_amount,hardship_length,hardship_dpd,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,settlement_amount,settlement_percentage,settlement_term
count,1.000000e+05,0.0,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,1.000000e+05,99998.000000,100000.000000,100000.000000,100000.000000,100000.000000,51806.000000,17804.000000,100000.000000,100000.000000,100000.000000,99963.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,1.000000e+05,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,29371.000000,100000.0,502.000000,500.000000,100000.000000,100000.000000,1.000000e+05,21372.000000,21372.000000,21372.000000,21372.000000,20810.000000,21372.000000,18617.000000,21372.000000,21372.000000,21372.000000,21372.000000,100000.00000,21372.000000,21372.000000,21372.000000,100000.000000,100000.000000,99018.000000,98965.000000,100000.000000,100000.000000,97222.000000,100000.00000,100000.000000,100000.000000,100000.000000,99057.000000,25427.000000,89230.000000,35910.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,99999.000000,100000.000000,100000.000000,94808.000000,100000.000000,100000.000000,100000.000000,100000.000000,98942.000000,100000.000000,100000.000000,1.000000e+05,1.000000e+05,100000.000000,1.000000e+05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,781.0,781.000000,781.0,781.000000,621.000000,781.000000,781.000000,2949.000000,2949.000000,2949.000000
mean,6.532613e+07,NaN,15055.861000,15055.861000,15047.107500,12.200478,434.553148,7.837135e+04,19.250957,0.349760,694.592800,698.592930,0.585750,34.059974,65.416423,11.949110,0.238520,17673.155580,52.498989,25.261490,1001.822710,1001.229872,15155.901258,15147.325155,12136.019317,2797.622687,2.057928e+00,220.201313,39.257429,4176.534533,678.473310,662.900200,0.021870,44.168431,1.0,110402.958008,18.249940,0.005920,267.823900,1.421636e+05,1.109021,2.928832,0.761651,1.674574,20.912686,36552.811389,71.580491,1.389060,2.975482,5887.979740,60.881995,34503.37689,0.943945,1.537058,2.234091,4.715170,13285.490560,10148.025975,61.007727,0.010520,14.510140,127.510965,187.11690,13.537530,7.806680,1.658080,24.796

In [14]:
categorical_cols = df.select_dtypes(include = ['object']).columns.tolist()
categorical_cols

['term',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'earliest_cr_line',
 'initial_list_status',
 'last_pymnt_d',
 'next_pymnt_d',
 'last_credit_pull_d',
 'application_type',
 'verification_status_joint',
 'hardship_flag',
 'hardship_type',
 'hardship_reason',
 'hardship_status',
 'hardship_start_date',
 'hardship_end_date',
 'payment_plan_start_date',
 'hardship_loan_status',
 'disbursement_method',
 'debt_settlement_flag',
 'debt_settlement_flag_date',
 'settlement_status',
 'settlement_date']

In [16]:
# Remove target from features

target_col = 'loan_status'

if target_col in categorical_cols:
    categorical_cols.remove(target_col)

In [18]:
# check length

print("Numerical Columns: ", len(numerical_cols))
print("Categorical Columns: ", len(categorical_cols))

Numerical Columns:  115
Categorical Columns:  35


#### **Business interpretation**

Numerical features represent the financial strength and credit behavior of a borrower, such as income, debt ratio, and credit history.

Categorical features represent borrower segments and loan characteristics, such as loan purpose, employment type, and home ownership.

Separating these features allows applying appropriate preprocessing techniques and building interpretable, production-ready models.

In [17]:
df['loan_status'].value_counts()

loan_status
Fully Paid            70288
Charged Off           17603
Current               11402
Late (31-120 days)      441
In Grace Period         199
Late (16-30 days)        66
Default                   1
Name: count, dtype: int64